In [8]:
import os 
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pertpy as pt
import scanpy as sc
from pathlib import Path 

plt.rcParams["figure.figsize"] = (3, 3)

In [9]:
def assign_synth_samples(adata):
    adata.obs["sample_id"] = None
    n_control, n_trt = np.unique(adata.obs["treatment"], return_counts=True)[1]
    ctr_samples = np.random.randint(0, 2, n_control)
    trt_samples = np.random.randint(2, 4, n_trt)
    
    adata.obs.loc[adata.obs.treatment==0, "sample_id"] = ctr_samples
    adata.obs.loc[adata.obs.treatment==1, "sample_id"] = trt_samples
    
    adata.obs["sample_id"] = adata.obs["sample_id"].astype("category")

In [10]:
seeds = [8, 42, 100]

In [11]:
data_path = Path("/home/icb/alessandro.palma/environment/scFM_density_estimation/project_folder/data/pbmc68k_oversampled/")
result_folder = Path("/home/icb/alessandro.palma/environment/scFM_density_estimation/project_folder/results/abundance_test_experiment/milo/")

In [12]:
for i, seed in enumerate(seeds):
    np.random.seed(seed)
    for adata_name in os.listdir(data_path):
        result_name = f"oversamp_{adata_name.split('_')[1].replace('.h5ad', '')}_{i}"
        # Read AnnData 
        adata = sc.read_h5ad(data_path / adata_name)
        # Assign synthetic replicates 
        assign_synth_samples(adata)
        # Assign string treatment labels
        adata.obs["treatment_label"] = adata.obs.treatment.astype(str)
        # Initialize Milo
        milo = pt.tl.Milo()
        mdata = milo.load(adata)
        # Create neighbors
        milo.make_nhoods(mdata["rna"], prop=0.1)
        # Count per sample  
        mdata = milo.count_nhoods(mdata, sample_col="sample_id")
        # Differential abundance analysis 
        milo.da_nhoods(mdata, design="~treatment_label", solver='pydeseq2')
        # Build neighbor graph 
        milo.build_nhood_graph(mdata)
        # Compute log-ratio
        nhoods = mdata["rna"].obsm["nhoods"]
        logFC = mdata["milo"].var["logFC"].values[:, None]
        per_cell_fc = nhoods.dot(logFC) / nhoods.sum(1)
        adata.obs["log_ratio"] = per_cell_fc

        del adata.uns["nhood_neighbors_key"]
        adata.write_h5ad(result_folder / (result_name + ".h5ad"))
        print("Results saved")

! Using X_pca as default embedding
Using None as control genes, passed at DeseqDataSet initialization


Fitting size factors...
... done in 0.00 seconds.

Fitting dispersions...
... done in 1.50 seconds.

Fitting dispersion trend curve...
... done in 0.09 seconds.

/home/icb/alessandro.palma/miniconda3/envs/scvi_env/lib/python3.11/site-packages/pydeseq2/dds.py:548: UserWarning: As the residual degrees of freedom is less than 3, the distribution of log dispersions is especially asymmetric and likely to be poorly estimated by the MAD.
  self.fit_dispersion_prior()
Fitting MAP dispersions...
... done in 1.40 seconds.

Fitting LFCs...
... done in 1.04 seconds.

Calculating cook's distance...
... done in 0.01 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 0.92 seconds.



Log2 fold change & Wald test p-value: treatment_label 1 vs 0
       baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
0     15.005785        2.845331  0.756994  3.758721  0.000171  0.002370
1     13.974741        0.074507  0.632110  0.117870  0.906171  0.997697
2      4.784133       -1.292315  1.111374 -1.162809  0.244907  0.803042
3     10.225602       -0.678546  0.754900 -0.898854  0.368730  0.976607
4     15.528176       -0.223804  0.613789 -0.364627  0.715390  0.997697
...         ...             ...       ...       ...       ...       ...
4736  11.356299       -0.181610  0.735061 -0.247069  0.804855  0.997697
4737  18.175984        3.686575  0.815345  4.521490  0.000006  0.000246
4738  16.925170        0.028713  0.585151  0.049069  0.960865  0.997697
4739   7.763748        0.560033  0.872572  0.641819  0.520991  0.997697
4740   5.944815       -0.420100  0.947511 -0.443372  0.657497  0.997697

[4741 rows x 6 columns]


/tmp/ipykernel_1792053/1156788240.py:25: RuntimeWarning: invalid value encountered in divide
  per_cell_fc = nhoods.dot(logFC) / nhoods.sum(1)


Results saved
! Using X_pca as default embedding


...

Log2 fold change & Wald test p-value: treatment_label 1 vs 0
       baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
0     15.716439        0.174596  0.735429  0.237407  0.812341  0.996266
1     13.993826        0.044993  0.744733  0.060414  0.951826  0.996266
2      4.655927        0.451029  0.938457  0.480607  0.630796  0.996266
3     10.067682       -0.650428  0.776405 -0.837744  0.402175  0.996266
4     15.478545       -0.611564  0.714885 -0.855471  0.392290  0.996266
...         ...             ...       ...       ...       ...       ...
4736  11.291949        0.629655  0.755119  0.833848  0.404367  0.996266
4737  19.161028       -0.456269  0.690399 -0.660877  0.508691  0.996266
4738  16.946812        0.285158  0.703607  0.405280  0.685272  0.996266
4739   7.868357       -0.746967  0.820581 -0.910290  0.362669  0.996266
4740   5.887252        0.478079  0.866221  0.551914  0.581007  0.996266

[4741 rows x 6 columns]
Results saved
! Using X_pca as default embedding


Fitting size factors...
... done in 0.00 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 1.01 seconds.

Fitting dispersion trend curve...
/home/icb/alessandro.palma/miniconda3/envs/scvi_env/lib/python3.11/site-packages/pydeseq2/dds.py:820: UserWarning: The dispersion trend curve fitting did not converge. Switching to a mean-based dispersion trend.
  self._fit_parametric_dispersion_trend(vst)
... done in 0.07 seconds.

/home/icb/alessandro.palma/miniconda3/envs/scvi_env/lib/python3.11/site-packages/pydeseq2/dds.py:548: UserWarning: As the residual degrees of freedom is less than 3, the distribution of log dispersions is especially asymmetric and likely to be poorly estimated by the MAD.
  self.fit_dispersion_prior()
Fitting MAP dispersions...
... done in 1.38 seconds.

Fitting LFCs...
... done in 1.31 seconds.

Calculating cook's distance...
... done in 0.00 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 0.95 seconds.

/tmp/ipykernel_1792053/1156788240.py:25: RuntimeWarning: invalid value encountered in divide
  pe

Log2 fold change & Wald test p-value: treatment_label 1 vs 0
       baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
0     15.617850        0.956337  0.741406  1.289895  0.197087  0.999801
1     14.066163       -0.030982  0.742844 -0.041707  0.966732  0.999801
2      4.728946       -0.858504  0.950225 -0.903474  0.366274  0.999801
3     10.108449       -0.423656  0.825511 -0.513205  0.607808  0.999801
4     15.683878       -0.878856  0.749287 -1.172923  0.240827  0.999801
...         ...             ...       ...       ...       ...       ...
4736  11.414636       -0.853814  0.796356 -1.072152  0.283652  0.999801
4737  19.149672        0.296312  0.716213  0.413720  0.679079  0.999801
4738  17.149793       -0.722301  0.739086 -0.977289  0.328426  0.999801
4739   7.879026        0.099712  0.820914  0.121465  0.903323  0.999801
4740   5.879134        0.660111  0.893563  0.738741  0.460064  0.999801

[4741 rows x 6 columns]
Results saved
